## Importing Packages

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import sys
from mutagen.mp3 import MP3

sys.path.append(os.path.abspath(".."))


from pathlib import Path
from pydub import AudioSegment
from IPython.display import Markdown, display, update_display
import AudioTools.audio_tools as ats
import LLMFeatures.LLMFeatures as llmf

# API configurations

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
AUDIO_MODEL = "gpt-4o-mini-transcribe"
openai_client = OpenAI(api_key=openai_api_key)

# Uploading AudioFiles

In [ ]:
audio_filename = Path("../Audios/reunionMartinForecastReview.mp3")
in_path = Path('../Audios/')
out_path = Path('../Audios/converted')

In [ ]:
os.listdir('../Audios')

# Converting to mp3

In [ ]:
ats.convert_to_mp3(input_path=audio_filename, out_dir=out_path)

# Audio Splitting into parts

In [ ]:
audio_filename

In [ ]:
file_name_str = str(audio_filename.stem) + '.mp3'
mp3audio_filename = str(audio_filename.with_name("converted")) + '/' + file_name_str

mp3AudioPartsPath = ats.split_audio_with_overlap(
    input_path=mp3audio_filename,
    chunk_ms=20 * 60 * 1000,
    overlap_ms=2000
)

mp3AudioPartsPath

In [ ]:
for i in mp3AudioPartsPath:
    audio = MP3(i)
    print(f"Name of file: {i.name}")
    print("bytes:", i.stat().st_size)
    print("MB:", i.stat().st_size / (1024**2))
    print("time duration in seconds:", audio.info.length)
    print("time duration in minutes:", audio.info.length / 60)

# Translation is done here

In [ ]:
## remember, the audio files are in mp3AudioPartsPath
mp3AudioPartsPath

In [ ]:
allTranscription, dicTranscription = llmf.transcript_all(mp3AudioPartsPath, model = openai_client)

In [ ]:
display(Markdown(allTranscription))

In [ ]:
audio_filename.stem

In [ ]:
file_name = audio_filename.stem + "_Transcription.txt"
file_name

In [ ]:
## folder path
folder_transcription = Path('../Transcriptions')

if not (folder_transcription.exists() and folder_transcription.is_dir()):
    folder_transcription.mkdir(parents=True, exist_ok=True)

In [ ]:
with open(f'../Transcriptions/{file_name}', "w", encoding = "utf-8") as f:
    f.write(allTranscription)